# 02 — Datasets: metadata, folds & EDA

Builds per-dataset and combined metadata (labels, folds), verifies the inter-patient split, characterizes each cohort (class balance, demographics), and produces the EDA figures.

**Inputs:** PTB-XL database + Ningbo headers under `data/raw/`.
**Outputs:** `data/processed/metadata_{ptbxl,ningbo,combined}.csv` + figures in `reports/figures/`.

> Step-2 narrative (batch-effect discovery, decisions, bugs): see `JOURNAL.md`.

### Block 2.0: Project paths

In [ ]:
import os

ROOT = r'.'
PTBXL_BASE  = os.path.join(ROOT, 'data', 'raw', 'ptbxl',
                           'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3')
NINGBO_ROOT = os.path.join(ROOT, 'data', 'raw', 'ningbo',
                           'a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0')
PROCESSED   = os.path.join(ROOT, 'data', 'processed')
PTBXL_PLUS  = os.path.join(ROOT, 'data', 'raw', 'ptbxl_plus')

# Re-run guard: if the metadata CSVs already exist, blocks 2.1/2.3/2.6 load them from disk
# and skip the (re)build (the Ningbo header scan is the slow step, ~40 s).
# Set FORCE_REBUILD = True to rebuild everything from scratch.
FORCE_REBUILD = False

print("=== Path check ===")
for name, p in [('PTBXL_BASE',PTBXL_BASE),('NINGBO_ROOT',NINGBO_ROOT),('PROCESSED',PROCESSED)]:
    print(f"  {'OK     ' if os.path.exists(p) else 'MISSING'} {name}")

### Block 2.1: Build PTB-XL metadata

In [ ]:
# Build metadata_ptbxl.csv. Label rule: scp_codes['WPW'] == 100.0 (certain WPW only;
# lower-likelihood mentions are excluded). Skipped if the file already exists.
import os, pandas as pd, ast
os.makedirs(PROCESSED, exist_ok=True)
PTB_META = os.path.join(PROCESSED, 'metadata_ptbxl.csv')

if os.path.exists(PTB_META) and not FORCE_REBUILD:
    print("metadata_ptbxl.csv already exists -> SKIPPED rebuild (set FORCE_REBUILD=True to rebuild).")
else:
    df = pd.read_csv(os.path.join(PTBXL_BASE, 'ptbxl_database.csv'), index_col='ecg_id')
    def is_wpw(s):
        try: return 1 if ast.literal_eval(s).get('WPW', 0) == 100.0 else 0
        except Exception: return 0
    def has_wpw(s):
        try: return 1 if 'WPW' in ast.literal_eval(s) else 0
        except Exception: return 0
    df['label'] = df['scp_codes'].apply(is_wpw)
    meta = pd.DataFrame({'ecg_id': df.index, 'patient_id': df['patient_id'].astype(int),
                         'label': df['label'], 'fold': df['strat_fold'].astype(int), 'source': 'ptbxl'})
    n_wpw = int((meta.label==1).sum()); n_any = int(df['scp_codes'].apply(has_wpw).sum())
    print(f"PTB-XL: {len(meta)} ECGs | WPW {n_wpw} | Not-WPW {int((meta.label==0).sum())}")
    print(f"WPW mentioned (any likelihood): {n_any} | excluded (<100): {n_any - n_wpw}")
    assert n_wpw == 70, f"expected 70 PTB-XL WPW, got {n_wpw}"
    meta.to_csv(PTB_META, index=False)
    print("Built metadata_ptbxl.csv")

### Block 2.2: PTB-XL inter-patient split check

In [ ]:
# Verify the PTB-XL inter-patient split (native strat_fold is patient-stratified).
import os, pandas as pd
meta = pd.read_csv(os.path.join(PROCESSED, 'metadata_ptbxl.csv'))

n_wpw = int((meta.label==1).sum()); n_not = int((meta.label==0).sum())
n_pat = meta[meta.label==1].patient_id.nunique()
wpw_by_fold = meta[meta.label==1].groupby('fold').size()
n_leak = int((meta[meta.label==1].groupby('patient_id')['fold'].nunique() > 1).sum())

print(f"PTB-XL: WPW {n_wpw} | Not-WPW {n_not} | ratio {n_not/n_wpw:.1f}:1 | unique WPW patients {n_pat}")
print(f"WPW per fold: {wpw_by_fold.to_dict()}")
print(f"WPW split -> Train(1-8) {int(wpw_by_fold.loc[1:8].sum())} | Val(9) {int(wpw_by_fold.get(9,0))} | Test(10) {int(wpw_by_fold.get(10,0))}")

assert (n_wpw, n_not) == (70, 21729), "unexpected PTB-XL class counts"
assert n_leak == 0, f"patient leak across folds: {n_leak}"
print("[OK] PTB-XL inter-patient split clean (0 patient across folds).")

### Block 2.3: Identify Ningbo WPW from headers

In [ ]:
# Identify Ningbo WPW from the SNOMED-CT #Dx field of each header, in parallel.
# The .hea scan is the slow step (~40 s) -> skipped if metadata_ningbo.csv already exists.
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import pandas as pd

NINGBO_META = os.path.join(PROCESSED, 'metadata_ningbo.csv')
ningbo_cached = os.path.exists(NINGBO_META) and not FORCE_REBUILD

if ningbo_cached:
    ningbo_df = pd.read_csv(NINGBO_META, dtype={'ecg_id': str})
    print(f"metadata_ningbo.csv already exists -> SKIPPED header scan (loaded {len(ningbo_df)} rows from cache).")
else:
    WPW_CODE = '74390002'  # SNOMED-CT code for WPW
    wfdb_root = os.path.join(NINGBO_ROOT, 'WFDBRecords')
    # No global recursive glob (would stall the kernel on 45k files): walk first-level folders.
    level1 = sorted(os.path.join(wfdb_root, d) for d in os.listdir(wfdb_root)
                    if os.path.isdir(os.path.join(wfdb_root, d)))
    hea_paths = []
    for d1 in tqdm(level1, desc="Listing .hea", unit="dir"):
        for dp, _, files in os.walk(d1):
            hea_paths += [os.path.join(dp, f) for f in files if f.endswith('.hea')]
    def read_hea(path):
        dx, age, sex = [], None, None
        try:
            with open(path) as f:
                for line in f:
                    if line.startswith('#Dx:'):   dx  = [c.strip() for c in line.replace('#Dx:','').strip().split(',')]
                    elif line.startswith('#Age:'): age = line.replace('#Age:','').strip()
                    elif line.startswith('#Sex:'): sex = line.replace('#Sex:','').strip()
        except Exception:
            pass
        return {'ecg_id': os.path.basename(path).replace('.hea',''),
                'label': 1 if WPW_CODE in dx else 0, 'age': age, 'sex': sex,
                'rel_path': os.path.relpath(path, NINGBO_ROOT).replace('.hea','')}
    with ThreadPoolExecutor(max_workers=16) as ex:
        ningbo_df = pd.DataFrame(list(tqdm(ex.map(read_hea, hea_paths),
                                           total=len(hea_paths), desc="Reading .hea", unit="f")))
    print(f"Ningbo: {len(ningbo_df)} ECGs | WPW {int((ningbo_df.label==1).sum())} | Not-WPW {int((ningbo_df.label==0).sum())}")

assert len(ningbo_df) == 45152, f"expected 45152 Ningbo records, got {len(ningbo_df)}"
assert int((ningbo_df.label==1).sum()) == 72, "expected 72 Ningbo WPW"

### Block 2.4: Stratified folds for Ningbo

In [ ]:
# Create 10 stratified folds for Ningbo. Ningbo = 1 ECG per patient, so a stratified record
# split IS a valid inter-patient split. Skipped if folds were already loaded from cache.
from sklearn.model_selection import StratifiedKFold
import numpy as np

if ningbo_cached and 'fold' in ningbo_df.columns:
    print("Folds already present in cached metadata -> SKIPPED fold creation.")
else:
    ningbo_df = ningbo_df.reset_index(drop=True)
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    ningbo_df['fold'] = -1
    for k, (_, idx) in enumerate(skf.split(np.zeros(len(ningbo_df)), ningbo_df.label.values), start=1):
        ningbo_df.loc[idx, 'fold'] = k

wpw_by_fold = ningbo_df[ningbo_df.label==1].groupby('fold').size()
print(f"Ningbo WPW per fold: {wpw_by_fold.to_dict()}")
print(f"WPW split -> Train(1-8) {int(wpw_by_fold.loc[1:8].sum())} | Val(9) {int(wpw_by_fold.get(9,0))} | Test(10) {int(wpw_by_fold.get(10,0))}")
assert (ningbo_df.fold == -1).sum() == 0, "some Ningbo records unassigned"
assert int(wpw_by_fold.sum()) == 72, "WPW lost during fold assignment"

### Block 2.5: Save Ningbo metadata

In [ ]:
# Save metadata_ningbo.csv (keeps age/sex for the demographics block). Skipped if loaded from cache.
import os
if ningbo_cached:
    print("metadata_ningbo.csv already present -> SKIPPED save.")
else:
    meta_ningbo = ningbo_df[['ecg_id','label','fold','rel_path','age','sex']].copy()
    meta_ningbo['patient_id'] = meta_ningbo['ecg_id']   # 1 ECG = 1 patient
    meta_ningbo['source'] = 'ningbo'
    meta_ningbo = meta_ningbo[['ecg_id','patient_id','label','fold','source','rel_path','age','sex']]
    assert len(meta_ningbo) == 45152 and int((meta_ningbo.label==1).sum()) == 72
    assert meta_ningbo.ecg_id.nunique() == len(meta_ningbo), "duplicate ecg_id"
    meta_ningbo.to_csv(os.path.join(PROCESSED, 'metadata_ningbo.csv'), index=False)
    print(f"Saved metadata_ningbo.csv: {len(meta_ningbo)} rows")

### Block 2.6: Merge into combined metadata

In [ ]:
# Merge into metadata_combined.csv. The patient-leak check + counts are validated on EVERY run
# (even when loaded from cache, to catch a stale file). Merge skipped if the file already exists.
import os, pandas as pd
COMB_META = os.path.join(PROCESSED, 'metadata_combined.csv')

if os.path.exists(COMB_META) and not FORCE_REBUILD:
    combined = pd.read_csv(COMB_META, dtype={'ecg_id': str})
    built = False
    print(f"metadata_combined.csv already exists -> SKIPPED merge (loaded {len(combined)} rows).")
else:
    ptbxl  = pd.read_csv(os.path.join(PROCESSED, 'metadata_ptbxl.csv'))
    ningbo = pd.read_csv(os.path.join(PROCESSED, 'metadata_ningbo.csv'))
    cols = ['ecg_id','patient_id','label','fold','source']
    p, n = ptbxl[cols].copy(), ningbo[cols].copy()
    p['ecg_id'] = p.ecg_id.astype(str);  n['ecg_id'] = n.ecg_id.astype(str)
    p['patient_id'] = 'ptbxl_' + p.patient_id.astype(str)   # prefix to avoid cross-dataset id collisions
    n['patient_id'] = n.patient_id.astype(str)
    combined = pd.concat([p, n], ignore_index=True)
    built = True

# Validation (runs in both cases)
n_wpw = int((combined.label==1).sum()); n_dup = int(combined.ecg_id.duplicated().sum())
wpw_fold = combined[combined.label==1].groupby('fold').size()
tr, va, te = int(wpw_fold.loc[1:8].sum()), int(wpw_fold.get(9,0)), int(wpw_fold.get(10,0))
leak = int((combined.groupby('patient_id')['fold'].nunique() > 1).sum())
print(f"Combined: {len(combined)} ECGs | WPW {n_wpw} | duplicated ecg_id {n_dup}")
print(combined.groupby('source').agg(n=('label','size'), wpw=('label','sum')).to_string())
print(f"WPW split -> Train(1-8) {tr} | Val(9) {va} | Test(10) {te}")
assert (len(combined), n_wpw) == (66951, 142), "unexpected combined totals"
assert n_dup == 0, "duplicate ecg_id across datasets"
assert (tr, va, te) == (115, 13, 14), f"unexpected WPW fold split {(tr,va,te)}"
assert leak == 0, f"patient leak across folds in combined: {leak}"

if built:
    combined.to_csv(COMB_META, index=False)
    print("[OK] Built & saved metadata_combined.csv | 0 patient leak across folds.")
else:
    print("[OK] Cached metadata_combined.csv validated | 0 patient leak across folds.")

### Block 2.7: Population demographics by dataset (sex, age)

In [ ]:
# Population demographics (sex, age) per dataset. These identity-level attributes characterize
# each cohort for the paper but are NEVER used as model features (they would confound dataset
# identity with the WPW label). WPW is congenital, so WPW cases are expected to skew younger.
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
FIG = os.path.join(ROOT, 'reports', 'figures'); os.makedirs(FIG, exist_ok=True)

# PTB-XL: age/sex from source DB. sex 0=Male, 1=Female. age==300 is the anonymization
# artifact for patients >89 (293 records) -> set to NaN.
ptb = pd.read_csv(os.path.join(PTBXL_BASE, 'ptbxl_database.csv'))[['ecg_id','age','sex']].copy()
ptb = ptb.merge(pd.read_csv(os.path.join(PROCESSED, 'metadata_ptbxl.csv'))[['ecg_id','label']], on='ecg_id')
ptb['sex'] = ptb['sex'].map({0:'Male', 1:'Female'})
ptb.loc[ptb['age'] >= 300, 'age'] = np.nan

# Ningbo: age/sex parsed from headers (sex may be 'Unknown', age may be missing).
nin = pd.read_csv(os.path.join(PROCESSED, 'metadata_ningbo.csv'))[['ecg_id','label','age','sex']].copy()
nin['age'] = pd.to_numeric(nin['age'], errors='coerce')

def demo(df, name):
    nrec = len(df); m = int((df.sex=='Male').sum()); f = int((df.sex=='Female').sum())
    a = df['age'].dropna()
    print(f"  {name:13s} n={nrec:6d} | M {m:5d} ({m/nrec*100:4.1f}%)  F {f:5d} ({f/nrec*100:4.1f}%)"
          f" | age median {a.median():.0f}  IQR {a.quantile(.25):.0f}-{a.quantile(.75):.0f}  range {a.min():.0f}-{a.max():.0f}")

print("=== Full cohort ==="); demo(ptb,'PTB-XL'); demo(nin,'Ningbo')
print("=== WPW cases only ==="); demo(ptb[ptb.label==1],'PTB-XL WPW'); demo(nin[nin.label==1],'Ningbo WPW')

bins = np.arange(0, 95, 5)

# Figure 1: sex distribution per dataset (% of ECGs)
fig, ax = plt.subplots(figsize=(6, 4))
sx = pd.DataFrame({'PTB-XL':[(ptb.sex=='Male').mean()*100,(ptb.sex=='Female').mean()*100],
                   'Ningbo':[(nin.sex=='Male').mean()*100,(nin.sex=='Female').mean()*100]},
                  index=['Male','Female'])
sx.plot(kind='bar', ax=ax, color={'PTB-XL':'#1f77b4','Ningbo':'#ff7f0e'})
ax.set_title('Sex distribution per dataset'); ax.set_ylabel('% of ECGs')
ax.set_xticklabels(['Male','Female'], rotation=0); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'demographics_sex_by_dataset.png'), dpi=150, bbox_inches='tight'); plt.show()

# Figure 2: age distribution per dataset (full cohort)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(ptb.age.dropna(), bins=bins, density=True, alpha=.55, label='PTB-XL', color='#1f77b4')
ax.hist(nin.age.dropna(), bins=bins, density=True, alpha=.55, label='Ningbo', color='#ff7f0e')
ax.set_title('Age distribution per dataset'); ax.set_xlabel('Age (years)'); ax.set_ylabel('Density'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'demographics_age_by_dataset.png'), dpi=150, bbox_inches='tight'); plt.show()

# Figure 3: age WPW vs non-WPW (both datasets pooled) -> congenital, younger
fig, ax = plt.subplots(figsize=(6, 4))
allage = pd.concat([ptb[['age','label']], nin[['age','label']]])
ax.hist(allage[allage.label==0].age.dropna(), bins=bins, density=True, alpha=.55, label='non-WPW', color='#94a3b8')
ax.hist(allage[allage.label==1].age.dropna(), bins=bins, density=True, alpha=.6, label='WPW', color='#d62728')
ax.set_title('Age: WPW vs non-WPW (pooled)'); ax.set_xlabel('Age (years)'); ax.set_ylabel('Density'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'demographics_age_wpw_vs_nonwpw.png'), dpi=150, bbox_inches='tight'); plt.show()

### Block 2.8: EDA figures

In [ ]:
# EDA figures: class balance, WPW per fold, and an example WPW ECG from each dataset.
import os, wfdb, numpy as np, pandas as pd, matplotlib.pyplot as plt
FIG = os.path.join(ROOT, 'reports', 'figures'); os.makedirs(FIG, exist_ok=True)
combined    = pd.read_csv(os.path.join(PROCESSED, 'metadata_combined.csv'))
ningbo_meta = pd.read_csv(os.path.join(PROCESSED, 'metadata_ningbo.csv'))

# Fig 1: class distribution per dataset (log scale)
fig, ax = plt.subplots(figsize=(8, 5))
pivot = combined.groupby(['source','label']).size().unstack(fill_value=0); pivot.columns = ['Not-WPW','WPW']
pivot.plot(kind='bar', ax=ax, color=['#bbbbbb','#d62728'], logy=True)
ax.set_title('Class distribution per dataset (log scale)'); ax.set_xlabel('Dataset'); ax.set_ylabel('Number of ECGs (log)')
ax.set_xticklabels(pivot.index, rotation=0)
for i, src in enumerate(pivot.index):
    ax.text(i, pivot.loc[src,'WPW'], str(pivot.loc[src,'WPW']), ha='center', va='bottom', color='#d62728', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'class_distribution_by_dataset.png'), dpi=150); plt.show()

# Fig 2: WPW per fold (stacked by dataset)
fig, ax = plt.subplots(figsize=(10, 5))
wpw_fold = combined[combined.label==1].groupby(['fold','source']).size().unstack(fill_value=0)
wpw_fold.plot(kind='bar', stacked=True, ax=ax, color={'ptbxl':'#1f77b4','ningbo':'#ff7f0e'})
ax.set_title('WPW distribution per fold (stratification)'); ax.set_xlabel('Fold'); ax.set_ylabel('Number of WPW')
ax.set_xticklabels(range(1, 11), rotation=0)
ax.axvline(7.5, color='green', linestyle='--', alpha=.5); ax.text(7.5, ax.get_ylim()[1]*.9, ' val|test', color='green')
ax.legend(title='Dataset'); plt.tight_layout(); plt.savefig(os.path.join(FIG, 'wpw_by_fold.png'), dpi=150); plt.show()

# Fig 3: example WPW ECG, PTB-XL vs Ningbo (lead II, 3 s)
ptbxl_wpw_id = pd.read_csv(os.path.join(PROCESSED, 'metadata_ptbxl.csv')).query('label==1').iloc[0]['ecg_id']
ningbo_wpw = ningbo_meta.query('label==1').iloc[0]
folder = f'{(int(ptbxl_wpw_id)//1000)*1000:05d}'
sig_p, _ = wfdb.rdsamp(os.path.join(PTBXL_BASE, 'records500', folder, f'{int(ptbxl_wpw_id):05d}_hr'))
sig_n, _ = wfdb.rdsamp(os.path.join(NINGBO_ROOT, ningbo_wpw['rel_path']))
fig, axes = plt.subplots(2, 1, figsize=(14, 6)); fs = 500
axes[0].plot(np.arange(3*fs)/fs, sig_p[:3*fs, 1], lw=.8, color='#1f77b4')
axes[0].set_title(f"WPW PTB-XL (ecg_id {ptbxl_wpw_id}) -- lead II")
axes[1].plot(np.arange(3*fs)/fs, sig_n[:3*fs, 1], lw=.8, color='#ff7f0e')
axes[1].set_title(f"WPW Ningbo ({ningbo_wpw['ecg_id']}) -- lead II")
for a in axes: a.set_xlabel('Time (s)'); a.set_ylabel('Amplitude (mV)'); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'wpw_ecg_comparison.png'), dpi=150); plt.show()
print("Saved 3 EDA figures to reports/figures/")